<!-- circuitkit-logo -->
<p align="center">
  <picture>
    <source media="(prefers-color-scheme: dark)" srcset="../../docs/assets/circuitkit-logo-white.png">
    <img src="../../docs/assets/circuitkit-logo-black.png" width="220" alt="CircuitKIT">
  </picture>
</p>

# 🔧 Applications — Prune, Quantize, Selective Finetune

Once you've discovered a circuit, CircuitKIT provides three intervention types:

| Application | What it does | Use case |
|-------------|-------------|----------|
| **Pruning** | Structurally remove low-importance nodes | Reduce model size |
| **Quantization** | Circuit-guided mixed-precision | Reduce memory footprint |
| **Selective Finetuning** | Identify components for targeted PEFT/LoRA | Efficient adaptation |

All three follow the same pattern: **Discover → Apply → (Export)**.

- **Model:** `google/gemma-2-2b-it` (~2B params)
- **Task:** IOI
- **Runtime:** ~15-20 min on Colab T4

---

## Setup

In [1]:
# ⚠️ Must be set before any CUDA context is created.
# If you've already run GPU cells in a prior notebook, restart the runtime first.
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

import warnings
warnings.filterwarnings('ignore', module='transformer_lens')
warnings.filterwarnings('ignore', module='lm_eval')
warnings.filterwarnings('ignore', message='.*pretrained.*model kwarg.*')

import gc
import torch
from circuitkit import Pipeline

print(f"CUDA: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
CUDA: NVIDIA A100 80GB PCIe


## Discover a Shared Circuit

All three applications operate on the same discovered circuit.

In [2]:
MODEL = "google/gemma-2-2b-it"

pipe = Pipeline(
    MODEL,
    task="ioi",
    output_dir="./results/applications",
)

pipe.discover(
    algorithm="eap-ig",
    level="node",
    sparsity=0.3,
    n_examples=256,
    batch_size=2,
    ig_steps=5,
)

print(pipe._circuit)
print(f"Top 5: {pipe._circuit.top_nodes(5)}")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b-it into HookedTransformer
06:26:20 • ==================================================
06:26:20 • Starting: Circuit Discovery
06:26:20 • ==================================================
06:26:20 • [1/4] Loading and validating configuration
06:26:20 ⚙ Configuration validated successfully.
06:26:20 ⚙ Config: EAP-IG on ioi task, node level, 30% sparsity
06:26:20 • [2/4] Setting up model (model=google/gemma-2-2b-it)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b-it into HookedTransformer
06:26:31 🔧 Model loading: 11.03s
06:26:31 🔧 Model: google/gemma-2-2b-it (3.2B params, cuda device)
06:26:31 • [3/4] Running discovery algorithm (algorithm=eap-ig)
06:26:31 • Starting EAP-IG discovery algorithm
06:26:31 • Generating new IOI data for gemma-2-2b-it (samples=256, seed=42)
06:26:32 • Saved generated data to cache: cache/ioi/ioi_gemma-2-2b-it_256_seed42.csv


100%|██████████| 128/128 [01:56<00:00,  1.10it/s]

06:28:29 • Converted 234 node scores from EAP-IG graph to CircuitKIT format.
06:28:29 • Saved unified CircuitScores → results/applications/eap-ig_google_gemma-2-2b-it_ioi_node_scores.json
06:28:29 • Saved 'scores' → results/applications/eap-ig_google_gemma-2-2b-it_ioi_node_scores.pt
06:28:29 • [4/4] Identifying nodes to prune
06:28:29 • 
--- Running Categorized Node Pruning (Scope: BOTH) ---
06:28:29 • Found 208 Attention Heads and 26 MLP layers to consider for pruning.
06:28:29 • Pruning 7 out of 26 MLP layers (30%).
06:28:29 • Targeting to prune 62 out of 208 total Attention Heads (30%).
06:28:29 • Actually pruned 62 Attention Heads.
06:28:29 •   - Layer 0: Pruned 4/8 heads.
06:28:29 •   - Layer 1: Pruned 4/8 heads.
06:28:29 •   - Layer 2: Pruned 4/8 heads.
06:28:29 •   - Layer 3: Pruned 2/8 heads.
06:28:29 •   - Layer 4: Pruned 4/8 heads.
06:28:29 •   - Layer 5: Pruned 3/8 heads.
06:28:29 •   - Layer 6: Pruned 3/8 heads.
06:28:29 •   - Layer 7: Pruned 4/8 heads.
06:28:29 •   - Layer

---
## Section A: Structural Pruning

Pruning removes the least-important nodes identified by circuit discovery. `pipe.prune()` creates a masked copy of the model (the original is preserved).

In [3]:
# Prune at 30% sparsity
pipe.prune(sparsity=0.3, scope="both")
print("Pruning complete.")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b-it into HookedTransformer
06:28:41 • Starting structural masking: eap-ig scores on google/gemma-2-2b-it (inplace=False)
06:28:42 • Masking 70 nodes: ['A9.1', 'A13.3', 'A7.6', 'A0.6', 'A6.0'] + 65 more
06:28:42 ✓ Masking complete. Effective sparsity: 5.01%
Pruning complete.


### Sparsity Sweep

Compare different sparsity levels to find the best accuracy-size trade-off.

In [4]:
import pandas as pd

sweep_rows = []
for s in [0.1, 0.2, 0.3, 0.5]:
    p = Pipeline(MODEL, task="ioi", output_dir=f"./results/applications/sweep_{int(s*100)}")
    p.discover(algorithm="eap-ig", n_examples=256, batch_size=2, ig_steps=5)
    p.prune(sparsity=s, scope="both")
    sweep_rows.append({
        "Sparsity": s,
        "Circuit Size": len(p._circuit),
        "Top Node": list(p._circuit.top_nodes(1).keys())[0] if p._circuit.scores else "—",
    })
    print(f"  Sparsity {s:.0%} done")

    # prune() loads self._model + creates self._pruned_model (two full model
    # copies on GPU). Release both before the next iteration to avoid OOM.
    p._model = None
    p._pruned_model = None
    del p
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

pd.DataFrame(sweep_rows)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b-it into HookedTransformer
06:28:53 • ==================================================
06:28:53 • Starting: Circuit Discovery
06:28:53 • ==================================================
06:28:53 • [1/4] Loading and validating configuration
06:28:53 ⚙ Configuration validated successfully.
06:28:53 ⚙ Config: EAP-IG on ioi task, node level, 30% sparsity
06:28:53 • [2/4] Setting up model (model=google/gemma-2-2b-it)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b-it into HookedTransformer
06:29:04 🔧 Model loading: 10.98s
06:29:04 🔧 Model: google/gemma-2-2b-it (3.2B params, cuda device)
06:29:04 • [3/4] Running discovery algorithm (algorithm=eap-ig)
06:29:04 • Starting EAP-IG discovery algorithm
06:29:04 • Loading cached IOI data for gemma-2-2b-it from cache/ioi/ioi_gemma-2-2b-it_256_seed42.csv


100%|██████████| 128/128 [01:58<00:00,  1.08it/s]

...

100%|██████████| 128/128 [01:56<00:00,  1.10it/s]

06:38:48 • Converted 234 node scores from EAP-IG graph to CircuitKIT format.
06:38:48 • Saved unified CircuitScores → results/applications/sweep_50/eap-ig_google_gemma-2-2b-it_ioi_node_scores.json
06:38:48 • Saved 'scores' → results/applications/sweep_50/eap-ig_google_gemma-2-2b-it_ioi_node_scores.pt
06:38:48 • [4/4] Identifying nodes to prune
06:38:48 • 
--- Running Categorized Node Pruning (Scope: BOTH) ---
06:38:48 • Found 208 Attention Heads and 26 MLP layers to consider for pruning.
06:38:48 • Pruning 7 out of 26 MLP layers (30%).
06:38:48 • Targeting to prune 62 out of 208 total Attention Heads (30%).
06:38:48 • Actually pruned 62 Attention Heads.
06:38:48 •   - Layer 0: Pruned 4/8 heads.
06:38:48 •   - Layer 1: Pruned 4/8 heads.
06:38:48 •   - Layer 2: Pruned 4/8 heads.
06:38:48 •   - Layer 3: Pruned 2/8 heads.
06:38:48 •   - Layer 4: Pruned 4/8 heads.
06:38:48 •   - Layer 5: Pruned 3/8 heads.
06:38:48 •   - Layer 6: Pruned 3/8 heads.
06:38:48 •   - Layer 7: Pruned 4/8 heads.
06

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b-it into HookedTransformer
06:39:00 • Starting structural masking: eap-ig scores on google/gemma-2-2b-it (inplace=False)
06:39:00 • Masking 117 nodes: ['A9.1', 'A13.3', 'A7.6', 'A0.6', 'A6.0'] + 112 more
06:39:01 ✓ Masking complete. Effective sparsity: 9.39%
  Sparsity 50% done


,Sparsity,Circuit Size,Top Node
0,0.1,69,A25.4
1,0.2,69,A25.4
2,0.3,69,A25.4
3,0.5,69,A25.4


### Export the Pruned Checkpoint

Save as a standard HuggingFace checkpoint directory.

In [5]:
ckpt_path = pipe.export("./results/applications/pruned_checkpoint")
print(f"Checkpoint saved: {ckpt_path}")

# pipe._pruned_model was needed for export; release it now along with
# pipe._model before Section B loads its own model via pipe_q.
# pipe._artifact_path (a string) is still needed by Section B — it survives.
pipe._model = None
pipe._pruned_model = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Checkpoint saved: ./results/applications/pruned_checkpoint


---
## Section B: Circuit-Guided Quantization

Mixed-precision quantization keeps the most important layers (by circuit score) at higher precision while aggressively quantizing the rest.

- `high_fraction=0.3` → top 30% of layers stay at full precision
- The rest are quantized to `bits` width

In [6]:
# Fresh pipeline for quantization (avoids conflicts with pruning state)
pipe_q = Pipeline.from_artifact(
    pipe._artifact_path,
    model_name=MODEL,
    task="ioi",
    output_dir="./results/applications/quantized",
)

pipe_q.quantize(bits=4, high_fraction=0.3, backend="quanto")
print("Quantization complete.")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

06:39:12 • Detected architecture: gemma2 → gemma family
06:39:12 • ✓ Validated gemma2 architecture
06:39:12 • [quant] Quantizing low-tier layers to quanto.qint4 (36 pattern(s)) …
06:39:13 • [quant] Freezing quantized weights …
06:39:13 ✓ [quant] Freeze complete.
06:39:13 • [quant] Guard OK: 126/126 QModule(s) frozen (packed low-bit weights).
Quantization complete.


In [7]:
# Export quantized checkpoint
q_ckpt = pipe_q.export("./results/applications/quantized_checkpoint", intervention="quantization")
print(f"Quantized checkpoint: {q_ckpt}")

# Release pipe_q's model before Section C loads its own.
pipe_q._model = None
pipe_q._pruned_model = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Quantized checkpoint: ./results/applications/quantized_checkpoint


---
## Section C: Selective Finetuning

Selective finetuning identifies the most important attention heads and MLP layers for targeted adaptation. The output is a `SelectionResult` that maps directly to PEFT/LoRA gradient masks.

- `top_fraction=0.15` → select the top 15% of components
- `scope="both"` → select from both attention and MLP

In [8]:
# Use the same artifact from the earlier discovery
pipe_sf = Pipeline.from_artifact(
    pipe._artifact_path,
    model_name=MODEL,
    task="ioi",
    output_dir="./results/applications/finetune",
)

result = pipe_sf.selective_finetune(top_fraction=0.15, scope="both")
print(type(result))

06:39:16 • [score_loader] Loading scores from: results/applications/eap-ig_google_gemma-2-2b-it_ioi_node_scores.pt
06:39:16 • [score_loader] Detected: algo='eap-ig'  level='node'
06:39:16 • 
06:39:16 •   SCORE LOADER SUMMARY
06:39:16 • ============================================================
06:39:16 •   Algorithm : eap-ig
06:39:16 •   Level     : node
06:39:16 • 
  Attention heads loaded : 208
    Score range : [0.0000, 0.8008]
    Mean        : 0.0388
06:39:16 •     Layers      : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]
06:39:16 • 
  MLP layers loaded : 26  (node-level)
    Score range : [0.0007, 0.1123]
    Mean        : 0.0341
06:39:16 • 

<class 'circuitkit.applications.selective_finetuning.selector.SelectionResult'>


In [9]:
# Inspect the selection result
print(f"Attention layers selected: {len(result.attn)}")
print(f"MLP layers selected: {len(result.mlp)}")

# Show which layers were selected
print("\nAttention layers:")
for layer_key in sorted(result.attn.keys()):
    projections = list(result.attn[layer_key].keys())
    print(f"  {layer_key}: projections = {projections}")

print("\nMLP layers:")
for layer_key in sorted(result.mlp.keys()):
    val = result.mlp[layer_key]
    desc = f"{len(val)} neurons" if val is not None else "all (no mask)"
    print(f"  {layer_key}: {desc}")

Attention layers selected: 18
MLP layers selected: 4

Attention layers:
  attn_0: projections = ['q', 'k', 'v', 'o']
  attn_11: projections = ['q', 'k', 'v', 'o']
  attn_12: projections = ['q', 'k', 'v', 'o']
  attn_14: projections = ['q', 'k', 'v', 'o']
  attn_16: projections = ['q', 'k', 'v', 'o']
  attn_17: projections = ['q', 'k', 'v', 'o']
  attn_18: projections = ['q', 'k', 'v', 'o']
  attn_19: projections = ['q', 'k', 'v', 'o']
  attn_2: projections = ['q', 'k', 'v', 'o']
  attn_20: projections = ['q', 'k', 'v', 'o']
  attn_21: projections = ['q', 'k', 'v', 'o']
  attn_22: projections = ['q', 'k', 'v', 'o']
  attn_23: projections = ['q', 'k', 'v', 'o']
  attn_24: projections = ['q', 'k', 'v', 'o']
  attn_25: projections = ['q', 'k', 'v', 'o']
  attn_5: projections = ['q', 'k', 'v', 'o']
  attn_6: projections = ['q', 'k', 'v', 'o']
  attn_8: projections = ['q', 'k', 'v', 'o']

MLP layers:
  mlp_17: all (no mask)
  mlp_18: all (no mask)
  mlp_8: all (no mask)
  mlp_9: all (no mask

### Using SelectionResult with PEFT/LoRA

The `SelectionResult` tells you *which* components to train. Integrating with a training loop looks like:

```python
# Conceptual — not a runnable training loop
for name, param in model.named_parameters():
    param.requires_grad = False  # freeze all

# Unfreeze only selected components
for layer_key, proj_dict in result.attn.items():
    layer_idx = int(layer_key.split('_')[1])
    for proj, indices in proj_dict.items():
        # Create a gradient mask for these specific indices
        ...
```

See `circuitkit.applications.selective_finetuning` for the full API.

---
## Summary

| Application | Method | Output | Next step |
|-------------|--------|--------|-----------|
| Pruning | `pipe.prune()` | Masked model | `pipe.export()` → HF checkpoint |
| Quantization | `pipe.quantize()` | Mixed-precision model | `pipe.export()` → HF checkpoint |
| Selective FT | `pipe.selective_finetune()` | `SelectionResult` | PEFT/LoRA gradient mask setup |

All three can reuse the same discovered circuit — no re-running discovery.